# Exploración del comportamiento eCommerce (REES46)

Este cuaderno presenta el análisis de octubre y noviembre de 2019. El escaneo de los CSV completos se ejecuta con **PySpark local** mediante `src/eda.py`; el cuaderno solo carga el resumen JSON compacto y no carga los eventos en Pandas.

Los datos son históricos. Las horas y fechas de las gráficas se expresan en UTC.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next((candidate for candidate in (Path.cwd(), *Path.cwd().parents) if (candidate / 'src' / 'eda.py').is_file() and (candidate / 'data' / 'raw').is_dir()), None)
if ROOT is None:
    raise RuntimeError('No se encontró la raíz del proyecto ecommerce-big-data.')

# Cambia a True solo cuando quieras volver a escanear ambos CSV completos.
RUN_FULL_SCAN = False
SUMMARY_PATH = ROOT / 'data' / 'output' / 'eda_summary.json'
if RUN_FULL_SCAN:
    command = [
        sys.executable, 'src/eda.py', '--input',
        'data/raw/2019-Oct.csv', 'data/raw/2019-Nov.csv',
        '--output', str(SUMMARY_PATH.relative_to(ROOT)),
    ]
    subprocess.run(command, cwd=ROOT, check=True)

with SUMMARY_PATH.open(encoding='utf-8') as stream:
    summary = json.load(stream)
combined = summary['combined']
print('Motor:', summary['engine'])
print('Periodo UTC:', combined['period_utc'])
print('Filas Spark:', f"{combined['record_count']:,}")

## Inventario de los archivos analizados

Los conteos de esta tabla fueron validados por el parser CSV de Spark. El ZIP no se vuelve a contar: contiene copias comprimidas de los dos CSV.

In [ ]:
monthly_table = [
    {
        'Archivo': item['file'],
        'Filas': item['record_count'],
        'Bytes': item['file_bytes'],
        'Desde UTC': item['period_utc']['min'],
        'Hasta UTC': item['period_utc']['max'],
        'Eventos': item['event_counts'],
    }
    for item in summary['files']
]
display(monthly_table)

## Evidencia resumida para las 5V

Los valores de volumen, calidad, frecuencia y métricas son calculados localmente. La guía del curso publica una estimación del conjunto completo de siete meses; este proyecto analiza los dos meses presentes en `data/raw/`.

In [ ]:
five_vs = [
    {'V': 'Volume', 'Evidencia': f"{combined['record_count']:,} filas; {combined['file_bytes'] / 1024**3:.2f} GiB sin comprimir"},
    {'V': 'Velocity', 'Evidencia': f"{combined['frequency']['observed_utc_hours']:,} horas con eventos; promedio {combined['frequency']['mean_events_per_observed_utc_hour']:,.0f} eventos/hora activa"},
    {'V': 'Variety', 'Evidencia': 'CSV tabular de 9 columnas y campos categóricos jerárquicos'},
    {'V': 'Veracity', 'Evidencia': f"{combined['null_counts']['category_code']:,} category_code nulos; {combined['duplicate_excess_rows']:,} filas excedentes duplicadas por huella SHA-256"},
    {'V': 'Value', 'Evidencia': f"Conversión vista-compra por sesión: {combined['funnel']['view_to_purchase_rate']:.2%}"},
]
display(five_vs)

## Composición de eventos por mes

La columna `remove_from_cart` está contemplada por la descripción del caso; si no aparece en los resultados de estos archivos, no se observaron eventos de ese tipo en el periodo analizado.

In [ ]:
event_types = sorted({event for item in summary['files'] for event in item['event_counts']})
months = [item['file'].replace('.csv', '') for item in summary['files']]
positions = list(range(len(months)))
width = 0.24
fig, ax = plt.subplots(figsize=(9, 4.5))
for index, event_type in enumerate(event_types):
    values = [item['event_counts'].get(event_type, 0) for item in summary['files']]
    offsets = [position + (index - (len(event_types) - 1) / 2) * width for position in positions]
    ax.bar(offsets, values, width=width, label=event_type)
ax.set_xticks(positions, months)
ax.set_ylabel('Eventos')
ax.set_title('Eventos observados por mes')
ax.legend()
ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
plt.show()

## Frecuencia por hora UTC

El promedio siguiente considera solo horas que tienen eventos; no es una tasa de llegada en tiempo real. Los archivos son históricos y la arquitectura recomendada es Batch.

In [ ]:
hour_rows = combined['frequency']['by_hour_of_day_utc']
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar([row['hour'] for row in hour_rows], [row['events'] for row in hour_rows], color='#287271')
ax.set_xticks(range(24))
ax.set_xlabel('Hora del día (UTC)')
ax.set_ylabel('Eventos acumulados')
ax.set_title('Distribución horaria de eventos')
ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
plt.show()

## Funnel secuencial por sesión

El denominador son sesiones con `user_session` no vacío y una vista con hora válida. El cuaderno cuenta carrito posterior a la vista y compra posterior al carrito, permitiendo eventos en el mismo segundo (`<=`).

In [ ]:
funnel = combined['funnel']
stage_labels = ['Sesiones con vista', 'Carrito después de vista', 'Compra después de carrito']
stage_counts = [funnel['view_sessions'], funnel['cart_after_view_sessions'], funnel['purchase_after_cart_sessions']]
fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar(stage_labels, stage_counts, color=['#2a9d8f', '#e9c46a', '#e76f51'])
ax.bar_label(bars, labels=[f"{value:,}" for value in stage_counts], padding=3)
ax.set_ylabel('Sesiones')
ax.set_title('Funnel secuencial observado')
ax.tick_params(axis='x', rotation=12)
ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
plt.show()
print('Vista → carrito:', f"{funnel['view_to_cart_rate']:.2%}")
print('Vista → compra:', f"{funnel['view_to_purchase_rate']:.2%}")
print('Carrito → compra:', f"{funnel['cart_to_purchase_rate']:.2%}")

## Categorías de compra

La suma de `price` por evento `purchase` es una aproximación basada en el registro; no equivale a ingresos netos, margen ni conciliación contable.

In [ ]:
top_categories = combined['purchases']['top_categories'][:10]
display(top_categories)
labels = [row['category_code'] for row in reversed(top_categories)]
counts = [row['purchase_event_count'] for row in reversed(top_categories)]
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(labels, counts, color='#457b9d')
ax.set_xlabel('Eventos purchase')
ax.set_title('Categorías con más eventos de compra')
ax.grid(axis='x', alpha=0.25)
fig.tight_layout()
plt.show()